# Manuscript figure: objective comparison on the bowtie

The 7×7 bowtie from `01_signed_areas.ipynb` introduces two folded cells via
opposing $x_1$-displacements at the central row. This notebook solves the
bowtie with full-grid SLSQP under three different objective functions and
compares the resulting deformations side-by-side.

Constraint is the same in every run: every two-triangle area
$T_p^{1}, T_p^{2}$ must exceed `THRESHOLD = 0.01` (the codebase default).
Only the **objective** changes. With $d = \phi - \phi_\mathrm{init}$:

- **$L_1$** (sparse): $\sum_i \sqrt{d_i^2 + \varepsilon^2}$ — smoothed $|d|$
  with $\varepsilon = 10^{-4}$. Rewards solutions where the residual is
  concentrated on a few pixels.
- **$L_2$** (Euclidean norm): $\sqrt{\sum_i d_i^2 + \varepsilon^2}$ —
  smoothed 2-norm.
- **$L_2^2$** (Euclidean *squared*): $\tfrac{1}{2}\sum_i d_i^2$ — the
  codebase default `dvfopt.core.objective.objective_euc`. Same minimiser as
  $L_2$ (one is a monotone function of the other) but a strictly convex
  quadratic with a linear gradient, so SLSQP converges cleanly.

$L_1$ and $L_2/L_2^2$ are expected to disagree on **where** the residual
sits; $L_2$ and $L_2^2$ should agree on the minimiser (modulo numerical
behaviour) since their objectives are monotone-related.

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, NonlinearConstraint

from dvfopt.jacobian import triangle_sign_areas2D

THRESHOLD = 0.01
EPS = 1e-4   # smoothing for L1 / L2 (kink at d = 0)

## Bowtie input field

Same definition as Figure 1 of `01_signed_areas.ipynb`.

In [ ]:
H = W = 7
phi_bowtie = np.zeros((2, H, W))   # channels [dy, dx]
xc, yc = W // 2, H // 2
phi_bowtie[1, yc, xc]     = +1.2   # dx at (xc,   yc)
phi_bowtie[1, yc, xc + 1] = -1.2   # dx at (xc+1, yc)

T1_in, T2_in = triangle_sign_areas2D(phi_bowtie)
n_neg_in = int(((T1_in <= 0) | (T2_in <= 0)).sum())
min_T_in = float(min(T1_in.min(), T2_in.min()))
print(f'bowtie input:  folded triangles = {n_neg_in},  min T = {min_T_in:+.3f}')

bowtie input:  folded triangles = 2,  min T = -0.700


## Three objectives, one solver

Each objective returns `(value, gradient)` so SLSQP can be called with
`jac=True`. The flat decision vector is $z = [\,d_x^\mathrm{flat};\, d_y^\mathrm{flat}\,]$
(matching the codebase's `phi` flattening convention). The triangle
constraint stacks $T^{(1)}$ and $T^{(2)}$ for every cell and requires each
entry $\ge$ `THRESHOLD`.

In [ ]:
def objective_l1(z, z_init):
    diff = z - z_init
    s = np.sqrt(diff * diff + EPS * EPS)
    return float(s.sum()), diff / s


def objective_l2(z, z_init):
    diff = z - z_init
    norm = float(np.sqrt(float(np.dot(diff, diff)) + EPS * EPS))
    return norm, diff / norm


def objective_l2_2(z, z_init):
    # Codebase default (dvfopt.core.objective.objective_euc).
    diff = z - z_init
    return 0.5 * float(np.dot(diff, diff)), diff


def pack(phi):
    return np.concatenate([phi[1].flatten(), phi[0].flatten()])   # [dx, dy]


def unpack(z):
    n = H * W
    return z[n:].reshape(H, W), z[:n].reshape(H, W)               # dy, dx


def triangle_constraint(z):
    dy, dx = unpack(z)
    T1, T2 = triangle_sign_areas2D(np.stack([dy, dx]))
    return np.concatenate([T1.flatten(), T2.flatten()])


def solve(phi_init, objective_fn, max_iter=3000):
    z_init = pack(phi_init)
    constraint = NonlinearConstraint(triangle_constraint,
                                      lb=THRESHOLD, ub=np.inf)
    t0 = time.time()
    res = minimize(lambda z: objective_fn(z, z_init),
                   z_init.copy(), jac=True, method='SLSQP',
                   constraints=[constraint],
                   options={'maxiter': max_iter, 'ftol': 1e-10, 'disp': False})
    dy, dx = unpack(res.x)
    return np.stack([dy, dx]), res, time.time() - t0


OBJECTIVES = [
    ('L1',   r'$L_1$',    objective_l1),
    ('L2',   r'$L_2$',    objective_l2),
    ('L2_2', r'$L_2^2$',  objective_l2_2),
]

results = {}
for key, _, fn in OBJECTIVES:
    phi_out, res, elapsed = solve(phi_bowtie, fn)
    Tsol = triangle_sign_areas2D(phi_out)
    diff = phi_out - phi_bowtie
    results[key] = {
        'phi': phi_out,
        'res': res,
        'elapsed': elapsed,
        'min_T': float(Tsol.min()),
        'n_neg_T': int((Tsol <= 0).sum()),
        'l1':   float(np.abs(diff).sum()),
        'l2':   float(np.linalg.norm(diff)),
        'l2_2': 0.5 * float(np.dot(diff.flatten(), diff.flatten())),
    }
    print(f'{key:>4s}:  ok={res.success!s:<5s}  nit={res.nit:4d}  '
          f't={1000 * elapsed:6.0f} ms  '
          f'ℓ₁={results[key]["l1"]:6.3f}  '
          f'ℓ₂={results[key]["l2"]:6.3f}  '
          f'½∥·∥²={results[key]["l2_2"]:6.3f}  '
          f'min T={results[key]["min_T"]:+.3f}')

  L1:  ok=True   nit= 260  t=  2108 ms  ℓ₁= 1.420  ℓ₂= 1.004  ½∥·∥²= 0.504  min T=+0.010
  L2:  ok=True   nit=  15  t=    98 ms  ℓ₁= 2.190  ℓ₂= 0.853  ½∥·∥²= 0.364  min T=+0.010
L2_2:  ok=True   nit=   7  t=    41 ms  ℓ₁= 2.190  ℓ₂= 0.853  ½∥·∥²= 0.364  min T=+0.010


## Figure

Top row: the input bowtie plus the three converged warped grids, with
folded cells (if any) outlined in dark blue. Bottom row: per-pixel
Euclidean residual $\|\phi - \phi_\mathrm{init}\|_2$ for each solution,
on a shared colour scale so $L_1$’s concentrated correction vs.
$L_2/L_2^2$’s spread is visible at a glance.

In [ ]:
def plot_warped_grid(ax, phi, title):
    dy, dx = phi[0], phi[1]
    Hh, Ww = dy.shape
    yy, xx = np.mgrid[:Hh, :Ww]
    gx = xx + dx
    gy = yy + dy
    # Faint reference grid
    for i in range(Hh):
        ax.plot(xx[i], yy[i], color='#e8e8e8', lw=0.5, zorder=1)
    for j in range(Ww):
        ax.plot(xx[:, j], yy[:, j], color='#e8e8e8', lw=0.5, zorder=1)
    # Warped grid (blue)
    for i in range(Hh):
        ax.plot(gx[i], gy[i], color='#1f4e8c', lw=1.1, zorder=2)
    for j in range(Ww):
        ax.plot(gx[:, j], gy[:, j], color='#1f4e8c', lw=1.1, zorder=2)
    # Outline folded cells (if any) in darker blue
    tri = triangle_sign_areas2D(phi)
    for cy, cx in np.argwhere(tri.min(axis=0) <= 0):
        px = [gx[cy, cx], gx[cy, cx + 1],
              gx[cy + 1, cx + 1], gx[cy + 1, cx], gx[cy, cx]]
        py = [gy[cy, cx], gy[cy, cx + 1],
              gy[cy + 1, cx + 1], gy[cy + 1, cx], gy[cy, cx]]
        ax.plot(px, py, color='#0d47a1', lw=1.6, zorder=3)
    ax.set_aspect('equal'); ax.invert_yaxis()
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title, fontsize=10.5)


fig, axes = plt.subplots(2, 4, figsize=(15.5, 7.6), layout='constrained')

# ----- Top row: warped grids -----
plot_warped_grid(axes[0, 0], phi_bowtie, '(a) Bowtie input')
for col, (key, label, _) in enumerate(OBJECTIVES, start=1):
    r = results[key]
    panel = chr(ord('a') + col)
    plot_warped_grid(axes[0, col], r['phi'], f'({panel}) {label} objective')

# ----- Bottom row: per-pixel residual heatmaps (shared colour scale) -----
axes[1, 0].axis('off')
residuals = {key: np.linalg.norm(results[key]['phi'] - phi_bowtie, axis=0)
             for key, _, _ in OBJECTIVES}
vmax = max(r.max() for r in residuals.values()) or 1.0
for col, (key, label, _) in enumerate(OBJECTIVES, start=1):
    ax = axes[1, col]
    im = ax.imshow(residuals[key], cmap='Reds', vmin=0, vmax=vmax,
                   interpolation='nearest')
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    panel = chr(ord('e') + (col - 1))
    active = int((residuals[key] > 1e-3).sum())
    ax.set_title(f'({panel}) per-pixel residual under {label}\n'
                 f'{active} active px', fontsize=10.5)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Bowtie corrected under three objectives '
             '(same triangle constraint, T ≥ 0.01)', fontsize=12)
plt.show()

## Save

In [ ]:
output_dir = 'output/objective_comparison'
os.makedirs(output_dir, exist_ok=True)
fig.savefig(os.path.join(output_dir, 'objective_comparison.pdf'),
            bbox_inches='tight')
fig.savefig(os.path.join(output_dir, 'objective_comparison.png'),
            dpi=300, bbox_inches='tight')
print(f'Saved figure to {os.path.abspath(output_dir)}')

Saved figure to C:\Users\Andy\Documents\GitHub\UCI-iGravi\deformation-field-processing\notebooks\manuscript\output\objective_comparison
